### Transformer training pipeline

Dataset

↓

Tokenization

↓

Vocabulary creation

↓

Encoding sentences → numbers

↓

Padding

↓

Transformer model(Encoder+Decoder)

↓

Loss (CrossEntropy)

↓

Training loop

↓

Prediction

In [2]:
# step-1 install libraries 
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset     # load_dataset is a function from Hugging Face's datasets library. Its job is to load datasets from various sources.
from torch.nn.utils.rnn import pad_sequence
import math


In [9]:
# step-2 Load HuggingFace dataset
 
# It's a dataset identifier with two parts:
# First part (opus_books): The dataset name/collection
# Second part (en-fr): The specific language pair (English→French)

# Load the "opus_books" dataset for English-French translation
# "opus_books" is a collection of book translations
# "en-fr" means English-French language pair

dataset = load_dataset("opus_books", "en-fr")
# This creates a DatasetDict with 'train' split containing English-French sentence pairs
# Each example looks like: {'translation': {'en': 'Hello', 'fr': 'Bonjour'}}
# dataset is a Dictionary with keys: ["train"]

# Take only first 5000 examples from training set (to keep things manageable)
# .select(range(5000)) picks rows with indices 0 to 4999

data = dataset["train"].select(range(5000))
#      ↑            ↑      ↑
#      |            |      └── Take first 5000 rows
#      |            └── Access the 'train' split
#      └── DatasetDict containing all splits
# Now 'data' has 5000 examples instead of the full dataset (which could be lakhs)

# Extract English sentences from each example
# [x["translation"]["en"] for x in data] means:
# For each item 'x' in 'data', go to x["translation"] then get ["en"] value
# .lower() converts all text to lowercase (for consistency)
input = [x["translation"]["en"].lower() for x in data]
#        ↑  ↑            ↑
#        |  |            └── Get 'en' (English) from translation
#        |  └── Get 'translation' dictionary from the row
#        └── Each item 'x' is one row from data

# Result: List of English sentences like ["hello", "how are you", "the cat", ...]

# Extract French translations similarly
# Same process but get ["fr"] value instead of "en"
targets = [x["translation"]["fr"].lower() for x in data]
# Result: List of French sentences like ["bonjour", "comment allez-vous", "le chat", ...]



In [7]:
print(dataset)
first=dataset["train"][1]
print(first)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
{'id': '1', 'translation': {'en': 'Alain-Fournier', 'fr': 'Alain-Fournier'}}


In [ ]:
# step-3 Build Vocabulary

# Define a function that creates a word-to-number mapping from sentences
def build_vocab(sentences):
    
    # Initialize vocabulary with 3 special tokens:
    # <pad> (0): Used to make all sentences same length by padding
    # <start> (1): Marks beginning of sentence for decoder
    # <end> (2): Marks end of sentence for decoder
    
    vocab = {"<pad>": 0, "<start>": 1, "<end>": 2}
    
    # Start numbering new words from index 3 (since 0,1,2 are taken)
    idx = 3
    
    # Loop through each sentence in our list
    for sent in sentences:
        # Split sentence into individual words (by spaces)
        for word in sent.split():
            # If this word is not already in vocabulary
            if word not in vocab:
                # Add word to vocabulary with current index number
                vocab[word] = idx
                # Increment index for next new word
                idx += 1
    
    # Return the complete vocabulary dictionary
    return vocab

# Build vocabulary for English sentences (source language)
# input = list of English sentences

eng_vocab = build_vocab(input)
# Result: {"<pad>":0, "<start>":1, "<end>":2, "hello":3, "world":4, ...}
# Build vocabulary for French sentences (target language)
# targets = list of French sentences

tgt_vocab = build_vocab(targets)
# Result: {"<pad>":0, "<start>":1, "<end>":2, "bonjour":3, "monde":4, ...}
# Create reverse vocabulary for English (number → word)
# This swaps keys and values from eng_vocab

eng_inv = {v: k for k, v in eng_vocab.items()}
# Result: {0:"<pad>", 1:"<start>", 2:"<end>", 3:"hello", 4:"world", ...}
# .items() gives you pairs, v:k swaps them - so numbers become keys and words become values!
# Create reverse vocabulary for French (number → word)

tgt_inv = {v: k for k, v in tgt_vocab.items()}
# Result: {0:"<pad>", 1:"<start>", 2:"<end>", 3:"bonjour", 4:"monde", ...}

# Get vocabulary sizes (number of unique words including special tokens)
eng_vocab_size = len(eng_vocab)  # Used for embedding layer dimensions
tgt_vocab_size = len(tgt_vocab)  # Used for output layer dimensions

In [ ]:
# step-4 

def encode(sentence,vocab):
    
# Takes a sentence (like "hello world") and vocabulary dictionary
# Returns a tensor (list of numbers) representing the sentence

   return torch.tensor([vocab[x] for x in sentence.split() if x in vocab])
   # 1. sentence.split() → ["hello", "world"]
   # 2. vocab[w] for each word → [3, 4] (if hello=3, world=4 in vocab)
   # 3. torch.tensor() → converts to PyTorch tensor
   
eng_data=[encode(s,eng_vocab) for s in input]
# For each English sentence in 'inputs', convert to numbers
# Result: List of tensors, one per sentence
# After encoding:
#src_data = [
    #tensor([3, 4]),        # "hello world"
    #tensor([5, 6, 7])      # "how are you"]

tgt_input = []   # Will store decoder INPUT sequences (with <start>)
tgt_output = []  # Will store decoder OUTPUT sequences (with <end>)

french_tokens=[encode(s,tgt_vocab) for s in targets]
# Result: List of tensors like [tensor([3,4]), tensor([5,6,7,8]), ...]
# Example: "bonjour monde" → tensor([3,4])

for tokens in french_tokens:
    
    tgt_input.append(torch.cat([torch.tensor([tgt_vocab["<start>"]]), tokens]))
    # tokens is already a tensor from encode() function
    # Decoder input: <start> + tokens
    # torch.cat() joins tensors together
    # cat = concatenate (join/combine) tensors together
    
    
    tgt_output.append(torch.cat([tokens, torch.tensor([tgt_vocab["<end>"]])]))
    # Decoder output: tokens + <end>
    # Result: tensor([3, 4, 2]) for "bonjour monde"
    
    
    
   

    

